# Лабораторная 4. A/B извлечения

Живой прогон: `python -m ml.prompts_v2.ab` из корня репозитория. Ячейки читают сохранённые результаты.

In [1]:
import json
from pathlib import Path

run = Path("lab4-run")
if not run.is_dir():
    run = Path("docs/experiments/lab4-run")
comparison = json.loads((run / "comparison.json").read_text(encoding="utf-8"))
for name, result in comparison["variants"].items():
    counts = {key: f"{result['metrics'][key]['k']}/{result['metrics'][key]['n']}"
              for key in ("M2", "M3", "M6", "E3", "S1")}
    print(name, counts, result["cost"])
print("Выбран:", comparison["selected"])
print("Пройдены пороги:", comparison["decisions"])

A_baseline {'M2': '6/6', 'M3': '0/6', 'M6': '3/20', 'E3': '6/6', 'S1': '1/1'} {'input_tokens': 5283, 'output_tokens': 2328, 'cost_rub': 0.0, 'attempts': 6, 'unknown_usage': 0}
B_prompt {'M2': '3/6', 'M3': '0/6', 'M6': '3/20', 'E3': '6/6', 'S1': '1/1'} {'input_tokens': 5517, 'output_tokens': 2102, 'cost_rub': 0.0, 'attempts': 6, 'unknown_usage': 0}
C_calendar {'M2': '2/6', 'M3': '0/6', 'M6': '3/20', 'E3': '6/6', 'S1': '1/1'} {'input_tokens': 5533, 'output_tokens': 2103, 'cost_rub': 0.0, 'attempts': 6, 'unknown_usage': 0}
D_compact {'M2': '4/6', 'M3': '0/6', 'M6': '2/20', 'E3': '6/6', 'S1': '1/1'} {'input_tokens': 2419, 'output_tokens': 2015, 'cost_rub': 0.0, 'attempts': 6, 'unknown_usage': 0}
Выбран: C_calendar
Пройдены пороги: {'B_vs_A': True, 'C_vs_B': True, 'D_vs_C': False}


In [2]:
replay = json.loads((run / "cache.json").read_text(encoding="utf-8"))
for row in replay["requests"]:
    u = row["usage"]
    print(row["mode"], "tokens:", u["input_tokens"] + u["output_tokens"],
          "cache_hit:", u["cache_hit"], "latency_ms:", round(row["latency_ms"], 3))
assert replay["requests"][-1]["event"] == replay["requests"][-2]["event"]

uncached tokens: 1190 cache_hit: False latency_ms: 5058.767
uncached tokens: 1190 cache_hit: False latency_ms: 4845.027
cold tokens: 1190 cache_hit: False latency_ms: 4836.293
hit tokens: 0 cache_hit: True latency_ms: 0.081


Внешняя API-плата равна нулю. Стоимость оборудования и электричества не измерена. Результат на шести примерах относится к dev-набору.